# Lab 3: Visual RL — Training an Agent to Play Doom

**Course:** Deep Reinforcement Learning · **Framework:** PyTorch + Stable Baselines3

## Learning Objectives

- Set up ViZDoom with the Gymnasium wrapper and understand visual RL observation and action spaces
- Build a preprocessing wrapper (grayscale + resize) compatible with SB3's `CnnPolicy`
- Train PPO on `VizdoomBasic-v0`: the agent must align with a stationary enemy and shoot
- Add temporal context with frame stacking (`VecFrameStack`) and compare convergence speed
- Scale to `VizdoomDefendCenter-v0` and analyse what makes visual RL harder than state-based RL
- Record and replay gameplay video directly in the notebook

## Setup

ViZDoom ships with prebuilt Linux wheels so no compilation is required on Colab.

In [ ]:
!pip install vizdoom stable-baselines3[extra] gymnasium imageio imageio-ffmpeg opencv-python-headless -q

import vizdoom.gymnasium_wrapper   # registers VizdoomXxx-v0 environments
import gymnasium as gym
import numpy as np
import cv2, os, imageio
import matplotlib.pyplot as plt
from IPython.display import HTML
from base64 import b64encode
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecFrameStack, VecMonitor
from stable_baselines3.common.callbacks import EvalCallback

print('ViZDoom:', __import__('vizdoom').__version__)
print('SB3:    ', __import__('stable_baselines3').__version__)

## Explore the Environment

ViZDoom wraps the Doom game engine. Each scenario is defined by a `.cfg` + `.wad` pair specifying the map layout, enemy types, available actions, and reward structure.

**VizdoomBasic-v0** — the simplest scenario: the agent faces a stationary monster in a rectangular room. Three discrete actions: move left, move right, shoot. Reward: +101 for a kill, −5 per wasted shot.

In [ ]:
env = gym.make('VizdoomBasic-v0', render_mode='rgb_array')
obs, info = env.reset()

print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space, f'({env.action_space.n} discrete actions)')
print()
print('Screen shape:    ', obs['screen'].shape)    # (H, W, C) RGB
print('Game variables:  ', obs['gamevariables'])   # e.g. [ammo remaining]

frame = env.render()
plt.figure(figsize=(7, 4))
plt.imshow(frame)
plt.title('VizdoomBasic-v0 — raw frame from Doom engine')
plt.axis('off')
plt.tight_layout()
plt.show()
env.close()

## Preprocessing Wrapper

SB3's `CnnPolicy` expects a single `uint8` image tensor `(H, W, C)`. The ViZDoom wrapper returns a dict `{'screen': ..., 'gamevariables': ...}`, so we need to:

1. **Extract** the screen buffer
2. **Convert to grayscale** — colour adds no useful signal for these tasks and triples the input size
3. **Resize to 84×84** — the standard resolution used in the DQN and PPO Atari papers

This reduces the input from ~230k values (240×320×3) to ~7k (84×84×1) — a 33× reduction.

In [ ]:
class DoomScreenWrapper(gym.ObservationWrapper):
    '''Extract screen, convert to grayscale, resize to (screen_size, screen_size, 1).'''

    def __init__(self, env, screen_size=84):
        super().__init__(env)
        self.screen_size = screen_size
        self.observation_space = gym.spaces.Box(
            low=0, high=255,
            shape=(screen_size, screen_size, 1),
            dtype=np.uint8,
        )

    def observation(self, obs):
        screen = obs['screen'] if isinstance(obs, dict) else obs
        gray   = cv2.cvtColor(screen, cv2.COLOR_RGB2GRAY)
        small  = cv2.resize(gray, (self.screen_size, self.screen_size),
                             interpolation=cv2.INTER_AREA)
        return small[:, :, np.newaxis]


def make_doom(env_id, n_envs=4, screen_size=84, seed=0, render_mode=None):
    '''Vectorised, monitored Doom environment ready for SB3.'''
    def _make():
        env = gym.make(env_id, render_mode=render_mode)
        return DoomScreenWrapper(env, screen_size=screen_size)
    return VecMonitor(make_vec_env(_make, n_envs=n_envs, seed=seed))


# Sanity check
test_env = make_doom('VizdoomBasic-v0', n_envs=1)
obs = test_env.reset()
print('Wrapped observation shape:', obs.shape)   # (1, 84, 84, 1) — batch x H x W x C
test_env.close()

## Exercise 1: Train PPO on VizdoomBasic-v0

A random agent almost never kills the monster — it must first move into alignment, *then* shoot, a coordination the agent must discover purely from pixels and sparse reward.

We use PPO with SB3's `CnnPolicy`: three conv layers followed by a 512-unit MLP head.

| Hyperparameter | Value | Rationale |
|---|---|---|
| `n_steps=512` | steps per env before update | short rollouts → frequent gradient updates |
| `ent_coef=0.01` | entropy bonus | encourages early exploration |
| `n_envs=8` | parallel environments | 8× environment throughput |

Training 300k steps takes roughly 10–15 minutes on Colab's T4 GPU.

In [ ]:
train_env = make_doom('VizdoomBasic-v0', n_envs=8, seed=0)
eval_env  = make_doom('VizdoomBasic-v0', n_envs=1, seed=42)

os.makedirs('/tmp/doom-basic', exist_ok=True)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path='/tmp/doom-basic/',
    log_path='/tmp/doom-basic/',
    eval_freq=10_000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=0,
)

model = PPO(
    policy='CnnPolicy',
    env=train_env,
    n_steps=512,
    batch_size=64,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    learning_rate=2.5e-4,
    verbose=1,
    seed=0,
)

print('CNN encoder:')
print(model.policy.features_extractor)
print()
model.learn(total_timesteps=300_000, callback=eval_cb, progress_bar=True)

train_env.close()
eval_env.close()
print('Training complete.')

In [ ]:
log  = np.load('/tmp/doom-basic/evaluations.npz')
ts   = log['timesteps']
mean = log['results'].mean(axis=1)
std  = log['results'].std(axis=1)

plt.figure(figsize=(8, 4))
plt.plot(ts, mean, color='steelblue', label='Mean eval return (10 episodes)')
plt.fill_between(ts, mean - std, mean + std, alpha=0.2, color='steelblue')
plt.axhline(97, color='gray', linestyle='--', label='Kill every episode (~97)')
plt.xlabel('Environment steps')
plt.ylabel('Episode return')
plt.title('PPO on VizdoomBasic-v0')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Record and display gameplay video
model_basic = PPO.load('/tmp/doom-basic/best_model')

base = gym.make('VizdoomBasic-v0', render_mode='rgb_array')
wrapped = DoomScreenWrapper(base)
obs, _ = wrapped.reset()

frames = []
for _ in range(400):
    rgb = base.render()
    if rgb is not None:
        frames.append(rgb)
    action, _ = model_basic.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = wrapped.step(int(action))
    if terminated or truncated:
        obs, _ = wrapped.reset()

wrapped.close()

imageio.mimsave('/tmp/doom_basic.mp4', frames, fps=35)
with open('/tmp/doom_basic.mp4', 'rb') as f:
    v = b64encode(f.read()).decode()
HTML('<video width="480" controls autoplay loop>'
     '<source src="data:video/mp4;base64,' + v + '" type="video/mp4">'
     '</video>')

**Analysis questions:**
- Does the agent reliably kill the monster? At roughly what training step did the return first rise above zero?
- Try `ent_coef=0.0` — does the agent converge faster or get stuck in a suboptimal policy?
- SB3's `CnnPolicy` uses a fixed three-layer CNN encoder. Would a deeper or wider CNN help here?

## Exercise 2: Frame Stacking

A single frame gives the agent no information about *motion* — it cannot tell which direction an enemy is moving, or whether it is itself rotating. **Frame stacking** concatenates the last $k$ frames along the channel dimension, giving the agent a short-term temporal memory without any recurrent architecture.

This was a key component of DeepMind's original DQN (Mnih et al., 2015): $k=4$ frames turned a partially-observable MDP into a nearly Markovian one for Atari games.

With stacking, the observation shape changes from `(84, 84, 1)` → `(84, 84, 4)`.

In [ ]:
def make_doom_stacked(env_id, n_envs=4, n_stack=4, seed=0):
    base = make_doom(env_id, n_envs=n_envs, seed=seed)
    return VecFrameStack(base, n_stack=n_stack)

train_stacked = make_doom_stacked('VizdoomBasic-v0', n_envs=8, n_stack=4, seed=0)
eval_stacked  = make_doom_stacked('VizdoomBasic-v0', n_envs=1, n_stack=4, seed=42)

print('Stacked obs shape:', train_stacked.observation_space.shape)   # (84, 84, 4)

os.makedirs('/tmp/doom-stacked', exist_ok=True)

eval_cb_stacked = EvalCallback(
    eval_stacked,
    best_model_save_path='/tmp/doom-stacked/',
    log_path='/tmp/doom-stacked/',
    eval_freq=10_000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=0,
)

model_stacked = PPO(
    policy='CnnPolicy',
    env=train_stacked,
    n_steps=512,
    batch_size=64,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    learning_rate=2.5e-4,
    verbose=0,
    seed=0,
)

model_stacked.learn(total_timesteps=300_000, callback=eval_cb_stacked, progress_bar=True)
train_stacked.close()
eval_stacked.close()

In [ ]:
log_base    = np.load('/tmp/doom-basic/evaluations.npz')
log_stacked = np.load('/tmp/doom-stacked/evaluations.npz')

plt.figure(figsize=(8, 4))
for log, label, color in [
    (log_base,    'Single frame (k=1)', 'steelblue'),
    (log_stacked, '4-frame stack (k=4)', 'coral'),
]:
    ts   = log['timesteps']
    mean = log['results'].mean(axis=1)
    std  = log['results'].std(axis=1)
    plt.plot(ts, mean, label=label, color=color)
    plt.fill_between(ts, mean - std, mean + std, alpha=0.2, color=color)

plt.xlabel('Environment steps')
plt.ylabel('Episode return')
plt.title('Frame stacking comparison — VizdoomBasic-v0')
plt.legend()
plt.tight_layout()
plt.show()

**Discussion:**
- On the Basic scenario the gap between single-frame and stacked is likely small — why? (Hint: the monster doesn't move.)
- Which ViZDoom scenarios would benefit *most* from temporal context, and why?

## Exercise 3: VizdoomDefendCenter-v0

The agent stands in a circular arena. Monsters spawn at the perimeter and walk toward the center. The agent can **rotate left**, **rotate right**, or **shoot** — but cannot translate.

The agent must track moving targets and shoot before being overwhelmed. This requires temporal reasoning (which direction is the nearest enemy heading?) and a coordinated rotate-and-fire policy that no random policy ever discovers.

In [ ]:
env_dc = gym.make('VizdoomDefendCenter-v0', render_mode='rgb_array')
obs_dc, _ = env_dc.reset()
frame_dc = env_dc.render()

print('Observation space:', env_dc.observation_space)
print('Action space:     ', env_dc.action_space, f'({env_dc.action_space.n} actions)')

plt.figure(figsize=(7, 4))
plt.imshow(frame_dc)
plt.title('VizdoomDefendCenter-v0 — enemies approaching from the perimeter')
plt.axis('off')
plt.tight_layout()
plt.show()
env_dc.close()

In [ ]:
train_dc = make_doom_stacked('VizdoomDefendCenter-v0', n_envs=8, n_stack=4, seed=0)
eval_dc  = make_doom_stacked('VizdoomDefendCenter-v0', n_envs=1, n_stack=4, seed=42)

os.makedirs('/tmp/doom-dc', exist_ok=True)

eval_cb_dc = EvalCallback(
    eval_dc,
    best_model_save_path='/tmp/doom-dc/',
    log_path='/tmp/doom-dc/',
    eval_freq=10_000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=0,
)

model_dc = PPO(
    policy='CnnPolicy',
    env=train_dc,
    n_steps=512,
    batch_size=64,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.005,       # slightly less entropy pressure on this harder task
    learning_rate=2.5e-4,
    verbose=0,
    seed=0,
)

model_dc.learn(total_timesteps=500_000, callback=eval_cb_dc, progress_bar=True)
train_dc.close()
eval_dc.close()
print('Training complete.')

In [ ]:
log_dc = np.load('/tmp/doom-dc/evaluations.npz')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, log, title, color in [
    (axes[0], log_base, 'VizdoomBasic-v0 (300k steps)',        'steelblue'),
    (axes[1], log_dc,   'VizdoomDefendCenter-v0 (500k steps)', 'coral'),
]:
    ts   = log['timesteps']
    mean = log['results'].mean(axis=1)
    std  = log['results'].std(axis=1)
    ax.plot(ts, mean, color=color)
    ax.fill_between(ts, mean - std, mean + std, alpha=0.2, color=color)
    ax.set_xlabel('Environment steps')
    ax.set_ylabel('Episode return')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# Record DefendCenter gameplay
model_dc = PPO.load('/tmp/doom-dc/best_model')

base_dc   = gym.make('VizdoomDefendCenter-v0', render_mode='rgb_array')
wrapped_dc = DoomScreenWrapper(base_dc)
obs, _ = wrapped_dc.reset()

n_stack   = 4
frame_buf = np.repeat(obs, n_stack, axis=-1)   # (84, 84, 4)

frames_dc = []
for _ in range(1000):
    rgb = base_dc.render()
    if rgb is not None:
        frames_dc.append(rgb)
    action, _ = model_dc.predict(frame_buf, deterministic=True)
    obs, _, terminated, truncated, _ = wrapped_dc.step(int(action))
    frame_buf = np.roll(frame_buf, -1, axis=-1)
    frame_buf[:, :, -1] = obs[:, :, 0]
    if terminated or truncated:
        obs, _ = wrapped_dc.reset()
        frame_buf = np.repeat(obs, n_stack, axis=-1)

wrapped_dc.close()

imageio.mimsave('/tmp/doom_dc.mp4', frames_dc, fps=35)
with open('/tmp/doom_dc.mp4', 'rb') as f:
    v = b64encode(f.read()).decode()
HTML('<video width="480" controls autoplay loop>'
     '<source src="data:video/mp4;base64,' + v + '" type="video/mp4">'
     '</video>')

**Analysis questions:**
- How does the DefendCenter return curve compare to Basic in terms of: steps to first positive reward, final return level, and training stability (variance)?
- Why does visual RL generally require far more environment steps than state-based RL for equivalent difficulty tasks?
- Which of the following would you expect to help most on DefendCenter, and why?
  - (a) Larger CNN encoder  (b) More frames in the stack  (c) Curiosity-based exploration bonus  (d) Prioritised experience replay

## Going Further

ViZDoom has progressively harder scenarios that build on the skills learned here:

| Scenario | Challenge | Key skill |
|---|---|---|
| `VizdoomHealthGathering-v0` | Navigate and collect health packs | Spatial exploration |
| `VizdoomMyWayHome-v0` | Reach a fixed goal through a maze | Long-horizon planning |
| `VizdoomPredictPosition-v0` | Shoot an enemy that moves behind cover | Prediction under uncertainty |
| `VizdoomDeathmatch-v0` | Full Deathmatch with all weapons | Multi-skill integration |

For much faster training on harder scenarios, look at **[Sample Factory](https://github.com/alex-petrenko/sample-factory)** — a high-throughput async RL framework with native ViZDoom support that achieves 100k+ FPS on a single machine, enabling full Deathmatch agents in hours rather than days.